# Smoke Test: CQHR vs TSCP_R With Quantile Scores

This notebook checks the new `CQHR` implementation against our `TSCP_R` method in the CQR experiment runner.

`CQHR` follows Algorithm 2 of Sampson and Chan (2024): fit lower/upper coordinate-wise quantile models, compute raw signed CQR scores on calibration points, rescale each coordinate score by the calibration base interval length relative to a reference coordinate, take one conformal quantile of the maximum rescaled score, then map that scalar adjustment back to each test point using its own base interval lengths.

In [1]:
import pandas as pd

from utility.exps import run_cqr_synthetic_experiment

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 120)

## Design

The run is intentionally tiny. It checks that:

- `CQHR` runs through the CQR pipeline without capped/shifted score transformations.
- `TSCP_R` still runs on capped nonnegative CQR scores.
- Both methods produce trial-level, summary-level, and coordinate-wise outputs.

In [14]:
result = run_cqr_synthetic_experiment(
    dim_list=[2],
    sample_list=[20],
    alpha_list=[0.1],
    trials=200,
    methods=['CQHR', 'TSCP_R'],
    score_transform='capped',
    base_interval_alpha=0.5,
    n_train_pool=1000,
    n_features=5,
    n_informative=5,
    quantile_model_params={
        'n_estimators': 100,
        'max_depth': 5,
        'learning_rate': 0.08,
        'min_samples_leaf': 5,
    },
)

summary = result.summary_results.sort_values(['method', 'score_transform']).reset_index(drop=True)
display(summary[['method', 'score_transform', 'test_coverage_avg', 'coverage_vol_avg', 'coverage_max_length_median', 'runtime_avg']])

,method,score_transform,test_coverage_avg,coverage_vol_avg,coverage_max_length_median,runtime_avg
0,CQHR,native,0.900550,12461.731247,66.864576,0.000030
1,TSCP_R,capped,0.898975,148.878188,13.294251,0.000688


In [15]:
coordinate_summary = result.coordinate_summary_results.sort_values(['method', 'coordinate']).reset_index(drop=True)
display(coordinate_summary[['method', 'score_transform', 'coordinate', 'coordinate_base_length_avg', 'coordinate_adjustment_avg', 'coordinate_length_avg']])

,method,score_transform,coordinate,coordinate_base_length_avg,coordinate_adjustment_avg,coordinate_length_avg
0,CQHR,native,1,2.661098,15.957590,34.576277
1,CQHR,native,2,2.797770,75.361951,153.521672
2,TSCP_R,capped,1,2.661098,4.426737,11.514572
3,TSCP_R,capped,2,2.797770,4.953671,12.705112


## Smoke Assertions

In [4]:
assert not result.trial_results.empty
assert not result.summary_results.empty
assert not result.coordinate_summary_results.empty
assert set(result.summary_results['method']) == {'CQHR', 'TSCP_R'}
assert set(result.summary_results[result.summary_results['method'].eq('CQHR')]['score_transform']) == {'native'}
assert set(result.summary_results[result.summary_results['method'].eq('TSCP_R')]['score_transform']) == {'capped'}
assert result.summary_results['test_coverage_avg'].between(0, 1).all()
assert (result.coordinate_summary_results['coordinate_length_avg'] >= 0).all()

print('CQHR smoke test passed.')

CQHR smoke test passed.
